# W03 — Data Contract: Ranking Signal Analysis Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook is the **data contract** for the Ranking Signal Analysis lane. Every claim is backed by a query. Run top to bottom before committing.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1–2. Data Contract

**What one row means:** One row of `fact_content_daily_performance` is one `(report_date × pseudonymized client × pseudonymized content item)` performance observation — how one page performed for one client on one day (impressions, clicks, position, CTR from that day's Search Console pull).

**Table(s) I'll use:** `fact_content_daily_performance` as the fact table, joined to `dim_content` (one row per content item — carries `word_count`, publish/first-seen date, and other static content attributes) on the content key. `dim_clients` only if I need to filter/describe the client panel.

I'm treating `fact_content_query_90d` as **out of scope** for this contract — see exclusion below.

**Time window:** `month=2026-03` — a mid-panel month, per the assignment's warning that the `_sample` (June 2026) is the sealed final month and must never be used to develop label logic.

**What I'd predict/rank (label or proxy):** Same target as my capstone framing — average search position for a content item, used as the ranking signal that drives the refresh-priority queue (striking-distance: position 11–20; top-10-low-CTR: position ≤10 with low CTR). For this month's slice, the *proxy* is `AVG(gsc_avg_position)` per content item over March, rolled up from the daily grain.

**One thing I deliberately exclude:** `fact_content_query_90d`. It has a different grain (client × content × query hash, over a fixed *90-day* rolling window with last-30/prev-30 sub-windows) that doesn't line up with a single calendar month — mixing it in here would silently blend two different time windows into one row. I'll bring it in later (if at all) as its own join, not folded into this contract.

## Setup — connect to the warehouse

Store your read token as a Colab Secret named `HF_TOKEN` (key icon in the left sidebar). Never paste a token directly into a cell — this repo is public.

In [ ]:
%pip install -q duckdb

import os
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found. Add it as a Colab Secret named HF_TOKEN.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
print("Connected to warehouse:", REL)

### Schema discovery — run this before writing any query

Do not assume column names. Confirm them here, then fix the queries below if `DESCRIBE` shows something different from what is used (e.g. the warehouse uses `client_hash_id` not `client_id`, `gsc_avg_position` not `position`, etc.).

In [ ]:
tables = {
    'fact_content_daily_performance (March partition)': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet') LIMIT 1",
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet') LIMIT 1",
}
for label, src in tables.items():
    print(f"\n=== {label} ===")
    display(con.sql(f"DESCRIBE SELECT * FROM {src}").df())

## 3a. Query 1 — grain check

Prove that one row really is one `(report_date × client × content item)`. Expect an **empty** result below — if it's not empty, the grain claim in section 1 is wrong and the claim (not the query) needs fixing.

> **Column-name note (confirmed by DESCRIBE above):** The warehouse uses `client_hash_id` and `content_hash_id`, not `client_id` / `content_id`.

In [ ]:
q1 = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()
print(f"Rows violating the (date, client, content) grain: {len(q1)}")
q1

## 3b. Query 2 — row count and date span for the slice

In [ ]:
q2 = con.sql(f"""
    SELECT
        COUNT(*)                         AS n_rows,
        MIN(report_date)                 AS first_date,
        MAX(report_date)                 AS last_date,
        COUNT(DISTINCT content_hash_id)  AS n_content_items,
        COUNT(DISTINCT client_hash_id)   AS n_clients
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
q2

## 3c. Query 3 — availability, filtered with `IS TRUE`

> **Column-name note (confirmed by DESCRIBE above):** The position column in the warehouse is `gsc_avg_position`. All queries use `gsc_avg_position`, `gsc_clicks`, and `gsc_impressions`.

In [ ]:
q3_total = con.sql(f"""
    SELECT COUNT(*) AS n_total
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

q3_available = con.sql(f"""
    SELECT COUNT(*) AS n_available
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE (gsc_avg_position IS NOT NULL) IS TRUE
""").df()

n_total     = q3_total['n_total'][0]
n_available = q3_available['n_available'][0]
print(f"Total rows: {n_total:,}")
print(f"Rows with usable position data: {n_available:,}")
print(f"Survival rate: {n_available / n_total:.1%}")

## Five features — built from the same March slice

**Five features — available at the decision moment because:**

1. `total_clicks_march` — knowable because it's a sum of *already-occurred* daily click counts within the window itself, not a future period.
2. `total_impressions_march` — same reasoning: a same-window aggregate of already-observed daily impressions.
3. `observed_ctr_march` — derived purely from clicks/impressions already in hand for the window; no forward-looking data used.
4. `word_count` — a static content attribute from `dim_content`, set independent of any March performance.
5. `avg_position_march` — **this is the label itself**, not a feature. Listed here because it's a column in the frame above, but it must never be used as a model input — see the trap below.

In [ ]:
features_df = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        AVG(f.gsc_avg_position)                                      AS avg_position_march,
        SUM(f.gsc_clicks)                                             AS total_clicks_march,
        SUM(f.gsc_impressions)                                        AS total_impressions_march,
        SUM(f.gsc_clicks) / NULLIF(SUM(f.gsc_impressions), 0)        AS observed_ctr_march,
        MAX(d.word_count)                                             AS word_count
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('{REL}/dim_content.parquet') d
        ON f.content_hash_id = d.content_hash_id
    WHERE (f.gsc_avg_position IS NOT NULL) IS TRUE
    GROUP BY 1, 2
""").df()

print(f"Rows in feature frame: {len(features_df):,}")
features_df.head()

## 4. The trap — add a label-derived column on purpose, then remove it

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = features_df.dropna().copy()
df["needs_refresh"] = (df["avg_position_march"] > 20).astype(int)  # the label

honest_features = ["total_clicks_march", "total_impressions_march", "observed_ctr_march", "word_count"]

X_honest = df[honest_features]
y = df["needs_refresh"]
X_train, X_test, y_train, y_test = train_test_split(
    X_honest, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_acc = accuracy_score(y_test, clf.predict(X_test))
print(f"Honest accuracy: {honest_acc:.3f}")

In [ ]:
# Now add ONE label-derived column on purpose
df["position_bucket"] = pd.cut(
    df["avg_position_march"], bins=[0, 10, 20, 1000], labels=[0, 1, 2]
).astype(int)

leaky_features = honest_features + ["position_bucket"]
X_leaky = df[leaky_features]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42, stratify=y
)

clf_leaky = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
leaky_acc = accuracy_score(y_test_l, clf_leaky.predict(X_test_l))
print(f"Leaky accuracy (with position_bucket): {leaky_acc:.3f}")
print("This jumps toward 1.0 because position_bucket is just a binned copy of the label.")

# DELETE the leaky column and keep the honest number
del df["position_bucket"]
print(f"Honest number to report going forward: {honest_acc:.3f}")

`position_bucket` is derived directly from `avg_position_march`, which is the same quantity `needs_refresh` was thresholded from — it's the label wearing a disguise. Accuracy jumps toward 1.0 not because the model found a real signal, but because it's reading the answer off a rephrased version of the question. Deleted; the honest accuracy above (from clicks/impressions/CTR/word_count only) is the number that survives into future weeks.

## 4b. Limitation

`fact_content_daily_performance` is an **unbalanced panel** — per-client history depth differs (`dim_clients.gsc_data_start` / `ga4_data_start`), so March 2026 doesn't necessarily contain a full month of data for every client in `dim_clients`. A content item with a short observation window this month will have a noisier `avg_position_march` than one observed all 31 days, and naive aggregation treats them identically. This is a real limitation of using a single calendar month as the analysis window, separate from the leakage issue above.

## 5. Self-check

- [x] Five plain-words contract answers filled in (section 1–2)
- [x] Ran the `DESCRIBE` cell and confirmed/fixed real column names throughout (`gsc_avg_position`, `gsc_clicks`, `gsc_impressions`, `client_hash_id`, `content_hash_id`)
- [x] Three verification queries run with outputs visible (grain, row count/date span, availability with `IS TRUE`)
- [x] Five-feature frame built, each with an "available when?" line
- [x] Deliberate-leak experiment shown (honest vs. leaky accuracy) and the leaky column removed
- [x] One named limitation of this slice
- [ ] Notebook executed top-to-bottom in Colab (not just written) before committing
- [ ] Committed at `work/notebooks/w03_data_contract.ipynb`